In [9]:
import pandas as pd

df = pd.read_csv("../data/cleaned_data.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [10]:
analysis_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


In [11]:
rfm.describe()

,Recency,Frequency,Monetary
count,4338.000000,4338.000000,4338.000000
mean,92.536422,4.272015,2054.266460
std,100.014169,7.697998,8989.230441
min,1.000000,1.000000,3.750000
25%,18.000000,1.000000,307.415000
50%,51.000000,2.000000,674.485000
75%,142.000000,5.000000,1661.740000
max,374.000000,209.000000,280206.020000


In [12]:
#time series data

time_series = df.groupby('InvoiceDate').agg({
    'Quantity': 'sum',
    'TotalPrice': 'sum'
}).reset_index()

time_series.head()

,InvoiceDate,Quantity,TotalPrice
0,2010-12-01 08:26:00,40,139.12
1,2010-12-01 08:28:00,12,22.20
2,2010-12-01 08:34:00,98,348.78
3,2010-12-01 08:35:00,3,17.85
4,2010-12-01 08:45:00,449,855.86


In [13]:
#adding exact time features

time_series['day_of_week'] = time_series['InvoiceDate'].dt.dayofweek
time_series['month'] = time_series['InvoiceDate'].dt.month

In [14]:
#lag and rolling features 

time_series['lag_7'] = time_series['Quantity'].shift(7)
time_series['rolling_mean_7'] = time_series['Quantity'].rolling(7).mean()

In [15]:
time_series= time_series.dropna()

In [17]:
import os

os.makedirs("../data", exist_ok=True)

# Save RFM
rfm.to_csv("../data/rfm.csv")

# Save Time Series
time_series.to_csv("../data/time_series.csv", index=False)

print(" rfm.csv and time_series.csv saved")

 rfm.csv and time_series.csv saved
